# Faraday MR — corrida exhaustiva en Colab (GPU)

Proyecto III (foreground galáctico): disco + brazos espirales + campo magnético
regular/turbulento, ray-tracing desde un observador interior, comparación
contra datos reales (Oppermann & Enßlin 2012, catálogo NVSS Taylor+2009,
Haslam 408MHz, Planck 30GHz) y calibración estadística de amplitud.

Este notebook corre el **perfil "exhaustivo"** (`N_BASE=256`, mapa de cielo
360x181, ver `config_fisica.py`): resuelve `ARM_WIDTH` con ~4 celdas en vez
de 1 (sub-resuelto en el perfil "rapido" por defecto, pensado para CPU).
Pensado para GPU (`use_gpu=True`, falla explícitamente si no hay una
disponible, no corre "en silencio" 100x más lento de lo esperado, ver
`faradaymr.backend.get_backend`).

**IMPORTANTE antes de correr:** este notebook clona la rama `V4-Foreground-CMB` del
repo. Si tenés cambios sin pushear en tu copia local, hacé el push primero
(o subí manualmente la carpeta a `/content/` y saltate la celda de clonado)
-Colab no ve el estado de tu disco local, solo lo que esté en GitHub (o lo
que subas vos).

## 1. Verificar GPU asignada

In [ ]:
!nvidia-smi


## 2. Clonar el repositorio

Editá `REPO_URL`/`BRANCH` si trabajás desde un fork o una rama distinta.

In [ ]:
import os

REPO_URL = "https://github.com/hacostah/Mapas-de-Rotacion-de-Faraday"
BRANCH = "V4-Foreground-CMB"
RUTA_REPO = "/content/Mapas-de-Rotacion-de-Faraday"

if not os.path.isdir(RUTA_REPO):
    !git clone --branch {BRANCH} {REPO_URL} {RUTA_REPO}
else:
    print(f"Ya existe {RUTA_REPO}, no se clona de nuevo (borralo si querés una copia limpia).")

%cd {RUTA_REPO}


## 3. Instalar dependencias

Incluye `cupy` -la build que coincide con el CUDA de la imagen de Colab
(detectado automáticamente); si falla la detección, instalá a mano la que
corresponda (ver https://docs.cupy.dev/en/stable/install.html).

In [ ]:
!pip install -q -r requirements.txt

import subprocess

salida_nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout
if "release 12" in salida_nvcc:
    paquete_cupy = "cupy-cuda12x"
elif "release 11" in salida_nvcc:
    paquete_cupy = "cupy-cuda11x"
else:
    paquete_cupy = "cupy-cuda12x"  # default razonable para runtimes recientes de Colab
    print(f"No se pudo detectar la version de CUDA por nvcc; se intenta {paquete_cupy} de todos modos.")

print(f"Instalando {paquete_cupy} ...")
!pip install -q {paquete_cupy}


## 4. Verificar que el backend GPU quedó activo

In [ ]:
from faradaymr import get_backend

xp = get_backend(use_gpu=True)  # explota fuerte si cupy/CUDA no están bien instalados
print("Backend activo:", xp.__name__)
assert xp.__name__ == "cupy", "El backend no es cupy: revisar la instalación de la celda anterior."


## 5. Descargar los datos observacionales reales

~530 MB en total (Oppermann+2012, catálogo NVSS Taylor+2009, Haslam 408MHz,
Planck 30GHz) -se guardan en `data/external/` (gitignored, no vienen en el
clon).

In [ ]:
!python data/external/descargar_datos.py


## 6. Activar el perfil de resolución "exhaustivo"

**Tiene que correr ANTES de cualquier `import config`/`import model`/
`import run`/etc.** (`config_fisica.py` lee la variable de entorno una sola
vez, al importarse la primera vez; si algo ya lo importó con el perfil por
defecto, hay que reiniciar el runtime de Colab y volver a correr desde acá).

In [ ]:
import os

os.environ["FARADAYMR_PERFIL_RESOLUCION"] = "exhaustivo"

import config_fisica as cfg_fisica

print("Perfil activo:", cfg_fisica.PERFIL_RESOLUCION)
print("N_BASE:", cfg_fisica.N_BASE, " N_L x N_B:", cfg_fisica.N_L, "x", cfg_fisica.N_B)
print("DL:", cfg_fisica.DL, " PIXEL_CHUNK_SIZE:", cfg_fisica.PIXEL_CHUNK_SIZE)


## 7. Correr la simulación (GPU, perfil exhaustivo)

`use_gpu=True` explícito -no `None`/auto-detección- para que si algo salió
mal en la instalación de cupy, falle acá con un error claro en vez de caer
en silencio a CPU y tardar mucho más de lo esperado.

In [ ]:
import time

from run import ejecutar_corrida

t0 = time.perf_counter()
rm_map, i_map, q_map, u_map = ejecutar_corrida(use_gpu=True, seed=0)
print(f"Corrida completa en {time.perf_counter() - t0:.1f} s")


## 8. Comparar contra datos reales (Oppermann, NVSS, Haslam, Planck)

In [ ]:
from compare_observaciones import ejecutar_comparacion

resumen = ejecutar_comparacion()


## 9. Re-calibrar la amplitud del campo (recomendado)

La calibración `B0_REGULAR`/`B0_TURBULENTO` actual (`config_fisica.py`,
factor 0.269) se ajustó contra el perfil "rapido"; con `ARM_WIDTH` mejor
resuelta acá (perfil "exhaustivo") el factor óptimo puede correrse -este
paso lo recalcula contra ESTA corrida. Si el nuevo `alpha_combinado` difiere
bastante de 1.0, hay que decidir a mano si vale la pena actualizar
`config_fisica.py` (no se sobreescribe automáticamente, es una decisión de
quien corre esto, no del script).

In [ ]:
from calibrar_amplitud_campo import calibrar

resultado_calibracion = calibrar()


## 10. Empaquetar y descargar los resultados

In [ ]:
import shutil

ruta_zip = shutil.make_archive("/content/resultados_faradaymr", "zip", "results")
print(f"Resultados empaquetados en {ruta_zip}")

try:
    from google.colab import files
    files.download(ruta_zip)
except ImportError:
    print("No estás en Colab: el zip queda en la ruta de arriba, bajalo a mano.")
